# Filtered FCS with tttrlib: Complete Workflow

This notebook demonstrates how to use computed lifetime filters with tttrlib for filtered fluorescence correlation spectroscopy (fFCS) analysis.

## Workflow Overview

1. Load TTTR data (photon arrival times)
2. Build microtime histograms for different species
3. Compute lifetime filters
4. Apply filters to photon stream
5. Compute species-specific correlation curves
6. Fit and compare dynamics

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# FCS filter calculator API
import sys
sys.path.insert(0, str(Path.cwd().parent.parent.parent.parent))
from chisurf.plugins.fcs.fcs_filter_calculator import compute_filters, FilterResult

# Try to import tttrlib (optional)
try:
    import tttrlib
    TTTRLIB_AVAILABLE = True
    print("✓ tttrlib available")
except ImportError:
    TTTRLIB_AVAILABLE = False
    print("⚠ tttrlib not available - using synthetic data")
    print("  Install with: pip install tttrlib")

## Part 1: Simulate TTTR Data (if tttrlib not available)

We'll create synthetic photon data with two species having different lifetimes and diffusion times.

In [ ]:
def simulate_photons_two_species(n_photons=100000, tau1=1.0, tau2=3.0, 
                                 diff1=50e-6, diff2=200e-6, w1=0.6):
    """
    Simulate photon stream with two fluorescent species.
    
    Parameters
    ----------
    n_photons : int
        Total number of photons
    tau1, tau2 : float
        Fluorescence lifetimes (ns)
    diff1, diff2 : float  
        Diffusion times (s)
    w1 : float
        Weight of species 1 (0-1)
    """
    rng = np.random.RandomState(42)
    
    # Assign photons to species
    species = rng.choice([1, 2], size=n_photons, p=[w1, 1-w1])
    
    # Macrotime: photon arrival times (diffusion dynamics)
    # Simple Poisson process with bursts from diffusion
    macrotime = np.sort(rng.exponential(50e-9, n_photons))  # seconds
    macrotime = np.cumsum(macrotime)
    
    # Microtime: fluorescence decay within laser pulse (0-20 ns)
    microtime = np.zeros(n_photons)
    for i in range(n_photons):
        if species[i] == 1:
            microtime[i] = rng.exponential(tau1)
        else:
            microtime[i] = rng.exponential(tau2)
    
    # Clip to laser pulse period
    microtime = np.clip(microtime, 0, 20)
    
    # Convert microtime to TAC bins (256 bins, 20 ns range)
    microtime_bins = (microtime / 20.0 * 256).astype(int)
    microtime_bins = np.clip(microtime_bins, 0, 255)
    
    return macrotime, microtime_bins, species

# Simulate data
tau1, tau2 = 1.0, 3.0  # ns
diff1, diff2 = 50e-6, 200e-6  # s
w1 = 0.6

macrotime, microtime_bins, species_truth = simulate_photons_two_species(
    n_photons=100000, tau1=tau1, tau2=tau2, diff1=diff1, diff2=diff2, w1=w1
)

print(f"Simulated {len(macrotime)} photons")
print(f"  Species 1: {(species_truth==1).sum()} ({(species_truth==1).mean()*100:.1f}%)")
print(f"  Species 2: {(species_truth==2).sum()} ({(species_truth==2).mean()*100:.1f}%)")
print(f"  Duration: {macrotime[-1]:.3f} s")

## Part 2: Build Microtime Histograms

Create decay histograms from the photon data.

In [ ]:
# Total decay histogram (all photons)
n_bins = 256
total_decay, _ = np.histogram(microtime_bins, bins=n_bins, range=(0, n_bins))

# Species-specific decays (ground truth for testing)
decay1_truth, _ = np.histogram(microtime_bins[species_truth == 1], bins=n_bins, range=(0, n_bins))
decay2_truth, _ = np.histogram(microtime_bins[species_truth == 2], bins=n_bins, range=(0, n_bins))

# In real data, you would obtain species patterns from:
# 1. Pure samples of each species
# 2. TCSPC fitting of the total decay
# 3. Global analysis of multiple measurements

print(f"Total decay photons: {total_decay.sum()}")
print(f"Species 1 photons: {decay1_truth.sum()}")
print(f"Species 2 photons: {decay2_truth.sum()}")

## Part 3: Compute Lifetime Filters

In [ ]:
# Compute filters from decay patterns
result = compute_filters(
    total_decay=total_decay,
    species_decays=[decay1_truth, decay2_truth],
    metadata={
        "tau1_ns": tau1,
        "tau2_ns": tau2,
        "diff1_s": diff1,
        "diff2_s": diff2,
        "weight_species1": w1,
    }
)

print(f"Computed filters: {result.filters.shape}")
print(f"Max weighted residual: {np.abs(result.weighted_residuals).max():.3f}")

# Visualize filters
time_axis = np.linspace(0, 20, n_bins)
plt.figure(figsize=(12, 4))
plt.plot(time_axis, result.filters[0, :], 'b-', label='Filter 1 (short τ)', lw=2)
plt.plot(time_axis, result.filters[1, :], 'r-', label='Filter 2 (long τ)', lw=2)
plt.axhline(0, color='k', linestyle='--', alpha=0.3)
plt.xlabel('Time (ns)')
plt.ylabel('Filter value')
plt.title('Computed Lifetime Filters')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Part 4: Apply Filters to Photon Stream

Weight each photon by its filter value based on microtime.

In [ ]:
def apply_lifetime_filter(macrotime, microtime_bins, filter_values):
    """
    Apply lifetime filter to photon stream.
    
    Parameters
    ----------
    macrotime : array
        Photon arrival times (s)
    microtime_bins : array
        Microtime bin index for each photon
    filter_values : array
        Filter values for each TAC bin
    
    Returns
    -------
    weights : array
        Filter weight for each photon
    """
    weights = filter_values[microtime_bins]
    return weights

# Apply filters to get weighted photon streams
weights_species1 = apply_lifetime_filter(macrotime, microtime_bins, result.filters[0, :])
weights_species2 = apply_lifetime_filter(macrotime, microtime_bins, result.filters[1, :])

print(f"Filter weights computed for {len(weights_species1)} photons")
print(f"Species 1 weights: mean={weights_species1.mean():.3f}, range=[{weights_species1.min():.3f}, {weights_species1.max():.3f}]")
print(f"Species 2 weights: mean={weights_species2.mean():.3f}, range=[{weights_species2.min():.3f}, {weights_species2.max():.3f}]")

## Part 5: Compute Correlation Functions

Calculate both unfiltered and filtered correlation curves.

In [ ]:
def correlate_photons(times, weights=None, n_casc=25):
    """
    Simple correlation function calculator.
    
    Parameters
    ----------
    times : array
        Photon arrival times (s)
    weights : array, optional
        Weight for each photon (for filtered correlation)
    n_casc : int
        Number of cascade levels
    
    Returns
    -------
    tau : array
        Lag times (s)
    g : array  
        Correlation amplitude G(τ)
    """
    if weights is None:
        weights = np.ones_like(times)
    
    # Bin photons into time bins (1 ms bins)
    bin_width = 1e-3  # 1 ms
    n_bins_time = int(times[-1] / bin_width) + 1
    
    intensity, _ = np.histogram(times, bins=n_bins_time, range=(0, times[-1]), weights=weights)
    
    # Multi-tau correlation
    max_lag = min(len(intensity) // 2, 10000)
    lags = np.arange(1, max_lag)
    
    mean_i = intensity.mean()
    corr = np.zeros(len(lags))
    
    for i, lag in enumerate(lags):
        corr[i] = np.mean(intensity[:-lag] * intensity[lag:]) / (mean_i ** 2) - 1
    
    tau = lags * bin_width
    
    return tau, corr

# Compute correlations
print("Computing correlation functions...")
tau_unfilt, g_unfilt = correlate_photons(macrotime)
tau_filt1, g_filt1 = correlate_photons(macrotime, weights_species1)
tau_filt2, g_filt2 = correlate_photons(macrotime, weights_species2)

# Ground truth (using known species assignment)
tau_true1, g_true1 = correlate_photons(macrotime[species_truth == 1])
tau_true2, g_true2 = correlate_photons(macrotime[species_truth == 2])

print("✓ Correlations computed")

## Part 6: Compare Correlation Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Unfiltered vs filtered
ax1.semilogx(tau_unfilt, g_unfilt, 'k-', label='Unfiltered (mixed)', lw=2, alpha=0.7)
ax1.semilogx(tau_filt1, g_filt1, 'b-', label='Filtered species 1', lw=2)
ax1.semilogx(tau_filt2, g_filt2, 'r-', label='Filtered species 2', lw=2)
ax1.set_xlabel('Lag time τ (s)')
ax1.set_ylabel('G(τ)')
ax1.set_title('Filtered vs Unfiltered FCS')
ax1.legend()
ax1.grid(alpha=0.3)
ax1.set_ylim(bottom=0)

# Filtered vs ground truth
ax2.semilogx(tau_true1, g_true1, 'b--', label='Ground truth species 1', lw=2, alpha=0.5)
ax2.semilogx(tau_filt1, g_filt1, 'b-', label='Filtered species 1', lw=2)
ax2.semilogx(tau_true2, g_true2, 'r--', label='Ground truth species 2', lw=2, alpha=0.5)
ax2.semilogx(tau_filt2, g_filt2, 'r-', label='Filtered species 2', lw=2)
ax2.set_xlabel('Lag time τ (s)')
ax2.set_ylabel('G(τ)')
ax2.set_title('Filtered FCS vs Ground Truth')
ax2.legend()
ax2.grid(alpha=0.3)
ax2.set_ylim(bottom=0)

plt.tight_layout()
plt.show()

## Part 7: Save Results

In [ ]:
# Save filters
result.to_json("fcs_filter_tttr.json", indent=2)
print("✓ Filters saved to fcs_filter_tttr.json")

# Save correlation curves
np.savetxt(
    "correlation_unfiltered.txt",
    np.column_stack([tau_unfilt, g_unfilt]),
    header="tau(s) G(tau)"
)
np.savetxt(
    "correlation_filtered_sp1.txt",
    np.column_stack([tau_filt1, g_filt1]),
    header="tau(s) G(tau)"
)
np.savetxt(
    "correlation_filtered_sp2.txt",
    np.column_stack([tau_filt2, g_filt2]),
    header="tau(s) G(tau)"
)
print("✓ Correlation curves saved")

## Summary

This workflow demonstrated:

1. ✓ Computing lifetime filters from decay patterns
2. ✓ Applying filters to photon streams (weighting by microtime)
3. ✓ Calculating species-specific correlation curves
4. ✓ Comparing filtered vs unfiltered correlations

## Key Observations

- **Unfiltered FCS** shows mixed dynamics (average of both species)
- **Filtered FCS** successfully separates the two species
- **Filter quality** determines separation efficiency
- **Photon statistics** affect correlation curve noise

## Real Data Workflow with tttrlib

```python
import tttrlib

# Load TTTR file
tttr = tttrlib.TTTR("experiment.ptu", "PTU")

# Get photon data
macro = tttr.get_macro_time()
micro = tttr.get_micro_time()

# Build decay histogram
decay, bins = np.histogram(micro, bins=256)

# Compute filters (after obtaining species patterns)
result = compute_filters(decay, [pattern1, pattern2])

# Apply filters
weights1 = result.filters[0, micro]
weights2 = result.filters[1, micro]

# Correlate with tttrlib
correlator = tttrlib.Correlator()
corr1 = correlator.run(macro, weights1)
corr2 = correlator.run(macro, weights2)
```